### Notebook 8: MCP -- Network-Hosted Tools

In all previous notebooks, tools were defined locally as Python functions and imported directly. MCP (Model Context Protocol) inverts this: the tools run on a separate server, and the agent discovers and calls them
over the network. The agent doesn't import any tool code, it connects to an MCP server, queries what tools are available, and proxies calls through the MCP protocol.

This is the tool-level equivalent of what A2A does at the agent level:
- **A2A:** agents on the network, discovered via agent cards
- **MCP:** tools on the network, discovered via the MCP `list_tools` method

The same 11 financial tools from our `tools/` folder are wrapped in an MCP server. The ADK agent connects to it via `McpToolset` and uses them as if they were local.

### 1. Imports and Configuration

In [ ]:
import sys
import os
import time
import subprocess
import logging
from pathlib import Path

from google.adk.agents   import Agent
from google.adk.sessions import InMemorySessionService
from google.adk.runners  import Runner
from google.genai         import types
from IPython.display     import display, Markdown

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / "environment" / ".env")

from config import APP_NAME, USER_ID, strip_emojis

logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt = "%H:%M:%S",
)
logging.getLogger("google.genai").setLevel(logging.ERROR)
log = logging.getLogger(__name__)

MODEL = "gemini-3-flash-preview"

### 2. Start MCP Server

The MCP server is a standalone Python process that exposes all 11 financial tools over HTTP/SSE. It uses FastMCP (from the `mcp` package) to handle the protocol details. The server auto-generates tool schemas
from the function signatures and docstrings, the same information ADK uses when tools are defined locally.

In [3]:
# -----------------------------------------------------------------------
# Start the MCP server as a background subprocess.
# It serves tools over SSE on port 8010.
# -----------------------------------------------------------------------

mcp_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "mcp" / "financial_tools_server.py"),
]

mcp_process = subprocess.Popen(
    mcp_cmd,
    cwd    = str(PROJECT_ROOT),
    stdout = subprocess.PIPE,
    stderr = subprocess.PIPE,
)

time.sleep(3)

if mcp_process.poll() is not None:
    stderr = mcp_process.stderr.read().decode()
    log.error("MCP server exited with code %d: %s",
              mcp_process.returncode, stderr[-500:])
else:
    log.info("MCP server running on port 8010 (pid=%d)", mcp_process.pid)

07:16:04  INFO      MCP server running on port 8010 (pid=47114)


### 3. Verify MCP Server

Before connecting via ADK, verify the server is up by hitting the SSE
endpoint directly.

In [4]:
import httpx

try:
    r = httpx.get("http://localhost:8010/sse", timeout=3)
    log.info("MCP server responding: status=%d", r.status_code)
except Exception as e:
    log.error("MCP server not reachable: %s", e)

07:16:08  INFO      HTTP Request: GET http://localhost:8010/sse "HTTP/1.1 200 OK"
07:16:11  ERROR     MCP server not reachable: timed out


### 4. Connect ADK Agent to MCP Tools

`McpToolset` is ADK's bridge to MCP servers. It:

1. Connects to the MCP server (via SSE in this case).
2. Calls `list_tools` to discover available tools.
3. Converts MCP tool schemas to ADK-compatible `BaseTool` instances.
4. Proxies tool calls from the agent to the MCP server transparently.

In [ ]:
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import SseConnectionParams

# -----------------------------------------------------------------------
# Create the McpToolset pointing at our financial tools server.
# The agent will discover all 11 tools automatically.
# -----------------------------------------------------------------------

mcp_toolset = McpToolset(
    connection_params=SseConnectionParams(
        url="http://localhost:8010/sse",
    ),
)

# -----------------------------------------------------------------------
# Agent using MCP tools instead of local imports.
# -----------------------------------------------------------------------

mcp_agent = Agent(
    model       = MODEL,
    name        = "mcp_financial_advisor",
    description = "Financial advisor using tools served over MCP.",
    instruction = (
        "You are a financial advisor. Use the available tools to answer "
        "the user's question with concrete numbers. Always show the key "
        "figures from the tool output in your response. "
        "Never use emojis. Be professional."
    ),
    tools = [mcp_toolset],
)

### 5. Run Helper

In [6]:
async def run_agent(agent, query):
    """Send a query and return the final response."""
    runner = Runner(
        agent           = agent,
        app_name        = APP_NAME,
        session_service = InMemorySessionService(),
    )
    session = await runner.session_service.create_session(
        app_name = APP_NAME,
        user_id  = USER_ID,
    )
    content = types.Content(
        role  = "user",
        parts = [types.Part(text=query)],
    )

    final_text   = None
    final_author = None

    async for event in runner.run_async(
        user_id     = USER_ID,
        session_id  = session.id,
        new_message = content,
    ):
        author = getattr(event, "author", "unknown")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_call") and part.function_call:
                    log.info("[%s] tool_call: %s(%s)",
                             author, part.function_call.name,
                             dict(part.function_call.args or {}))
                if hasattr(part, "function_response") and part.function_response:
                    log.info("[%s] tool_response: %s -> %s",
                             author, part.function_response.name,
                             str(part.function_response.response)[:200])

        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if not hasattr(part, "text") or not part.text:
                    continue
                if not getattr(part, "thought", False):
                    final_text   = strip_emojis(part.text)
                    final_author = author

    if final_text:
        display(Markdown(f"**{final_author}:**\n\n{final_text}"))

    return final_text

### 6. Test Queries

Same queries from notebook 2, but now the tools are served over MCP
instead of imported locally. The results should be identical.

In [ ]:
# -----------------------------------------------------------------------
# Single tool call: mortgage payment calculation.
# The agent discovers calculate_monthly_payment from the MCP server.
# -----------------------------------------------------------------------

response = await run_agent(
    mcp_agent,
    "What would the monthly payment be on a 350k house with 70k down at 6.25% for 30 years?",
)

07:17:18  INFO      HTTP Request: GET http://localhost:8010/sse "HTTP/1.1 200 OK"
07:17:18  INFO      HTTP Request: POST http://localhost:8010/messages/?session_id=a888e1d6f48141629959cc408a4bef34 "HTTP/1.1 202 Accepted"
07:17:18  INFO      HTTP Request: POST http://localhost:8010/messages/?session_id=a888e1d6f48141629959cc408a4bef34 "HTTP/1.1 202 Accepted"
07:17:18  INFO      HTTP Request: POST http://localhost:8010/messages/?session_id=a888e1d6f48141629959cc408a4bef34 "HTTP/1.1 202 Accepted"
/Users/ciprian-ifrim/miniconda3/envs/adk-env/lib/python3.13/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.BASE_AUTHENTICATED_TOOL is enabled.
  check_feature_enabled()
07:17:18  INFO      
Note: Conversion of fields that are not included in the JSONSchema class are ignored.
Json Schema is now supported natively by both Vertex AI and Gemini API. Users
are recommended to pass/receive Json Schema directly to/from the API. For example:
1. 

**mcp_financial_advisor:**

For a $350,000 property with a $70,000 down payment, the loan amount is $280,000. At a 6.25% annual interest rate over a 30-year term, the monthly principal and interest payment is $1,724.01.

Over the full term of the loan, the total amount paid will be $620,642.94, which includes $340,642.94 in total interest. The loan-to-value ratio for this transaction is 80%.

07:18:17  INFO      HTTP Request: POST http://localhost:8010/messages/?session_id=a888e1d6f48141629959cc408a4bef34 "HTTP/1.1 202 Accepted"
07:18:19  INFO      HTTP Request: POST http://localhost:8010/messages/?session_id=a888e1d6f48141629959cc408a4bef34 "HTTP/1.1 202 Accepted"
07:18:19  INFO      HTTP Request: POST http://localhost:8010/messages/?session_id=a888e1d6f48141629959cc408a4bef34 "HTTP/1.1 202 Accepted"
07:18:29  INFO      HTTP Request: POST http://localhost:8010/messages/?session_id=a888e1d6f48141629959cc408a4bef34 "HTTP/1.1 202 Accepted"
07:18:30  INFO      HTTP Request: POST http://localhost:8010/messages/?session_id=a888e1d6f48141629959cc408a4bef34 "HTTP/1.1 202 Accepted"
07:18:54  ERROR     Error in sse_reader
Traceback (most recent call last):
  File "/Users/ciprian-ifrim/miniconda3/envs/adk-env/lib/python3.13/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/Users/ciprian-ifrim/miniconda3/envs/adk-env/lib/python3.13/si

In [8]:
# -----------------------------------------------------------------------
# Multi-tool query: affordability + eligibility.
# Both tools are discovered from the same MCP server.
# -----------------------------------------------------------------------

response = await run_agent(
    mcp_agent,
    "I earn 110k/year, have 400/month in debts, credit score 730. "
    "What is the max home I can afford at 6.5%? Am I eligible for a conventional loan "
    "if I want to buy a 450k house with 50k down?",
)

07:18:17  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
07:18:19  INFO      Response received from the model.
07:18:19  INFO      [mcp_financial_advisor] tool_call: calculate_affordability({'annual_rate': 6.5, 'monthly_debts': 400, 'annual_income': 110000})
07:18:19  INFO      [mcp_financial_advisor] tool_call: check_loan_eligibility({'credit_score': 730, 'property_value': 450000, 'annual_income': 110000, 'loan_amount': 400000, 'monthly_debts': 400})
07:18:19  INFO      [mcp_financial_advisor] tool_response: calculate_affordability -> {'content': [{'type': 'text', 'text': '{\n  "annual_income": 110000.0,\n  "monthly_debts": 400.0,\n  "max_monthly_mortgage_payment": 3541.67,\n  "max_loan_amount": 560329.99,\n  "max_home_price": 5603
07:18:19  INFO      [mcp_financial_advisor] tool_response: check_loan_eligibility -> {'content': [{'type': 'text', 'text': '{\n  "applicant_summary": {\n    "credit_score": 730,\n    "estima

**mcp_financial_advisor:**

Based on your annual income of $110,000 and monthly debts of $400, here is your financial assessment:

**Maximum Affordability**
At a 6.5% interest rate and a standard 43% debt-to-income ratio, your maximum affordability metrics are as follows:
*   **Maximum Monthly Mortgage Payment:** $3,541.67
*   **Maximum Home Price:** $560,329.99 (assuming 0% down)
*   **Maximum Loan Amount:** $560,329.99

**Conventional Loan Eligibility**
For a $450,000 home with a $50,000 down payment (a $400,000 loan), you are eligible for a conventional loan.
*   **Loan-to-Value (LTV) Ratio:** 88.89%
*   **Estimated Debt-to-Income (DTI) Ratio:** 31.94%
*   **Eligibility Status:** Qualified
*   **Requirements:** Because your down payment is less than 20%, Private Mortgage Insurance (PMI) will be required.

Your credit score of 730 and the resulting DTI of approximately 32% place you well within the qualifying range for conventional financing for a $450,000 property.

In [9]:
# -----------------------------------------------------------------------
# Investment projection: different tool category, same MCP server.
# -----------------------------------------------------------------------

response = await run_agent(
    mcp_agent,
    "If I invest 25k now and add 600/mo for 15 years at 7%, what will I have?",
)


07:18:29  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
07:18:30  INFO      Response received from the model.
07:18:30  INFO      [mcp_financial_advisor] tool_call: calculate_compound_interest({'years': 15, 'annual_rate': 7, 'monthly_contribution': 600, 'principal': 25000})
07:18:30  INFO      [mcp_financial_advisor] tool_response: calculate_compound_interest -> {'content': [{'type': 'text', 'text': '{\n  "principal": 25000.0,\n  "annual_rate": 7.0,\n  "years": 15,\n  "monthly_contribution": 600.0,\n  "future_value": 261401.05,\n  "total_contributions": 13300
07:18:30  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
07:18:32  INFO      Response received from the model.


**mcp_financial_advisor:**

Based on your investment plan, here is the projected growth of your portfolio after 15 years:

*   **Future Value:** $261,401.05
*   **Total Contributions:** $133,000.00
*   **Total Interest Earned:** $128,401.05

This calculation assumes an initial investment of $25,000.00 with monthly contributions of $600.00 at an annual return rate of 7%, compounded monthly. By the end of the 15-year period, your interest earned will nearly match your total contributions.

### 7. Comparison: Local Tools vs MCP Tools

| Aspect | Local tools (notebooks 2-3) | MCP tools (this notebook) |
|--------|---------------------------|--------------------------|
| Tool location | Imported Python functions in the same process | Separate server, accessed over HTTP/SSE |
| Discovery | Explicit import list | Automatic via `list_tools` protocol method |
| Schema source | Function signature + docstring | Same, but served via MCP protocol |
| Agent code change | `tools = [func1, func2]` | `tools = [McpToolset(...)]` |
| Performance | Direct function call | Network round-trip per tool call |
| Use case | Single-process agents | Shared tool servers, microservices, cross-language |

### 8. Shutdown

In [10]:
mcp_process.terminate()
mcp_process.wait(timeout=5)
log.info("MCP server stopped (pid=%d)", mcp_process.pid)

07:18:54  INFO      MCP server stopped (pid=47114)
